> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 07 · MCP and AgentCore Gateway: shared tools on Lambda

**CampusMate v4 (agent + cloud tools)**  ·  ⏱️ Time budget: **15 min theory + 30 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

Your two tools live inside your agent. Real campuses have shared systems (attendance, fees, exams) that many apps need. **AgentCore Gateway** turns an **AWS Lambda** function into a tool any agent can discover and call over **MCP**.

### What you'll do
- Understand MCP in five minutes
- Create the `check_attendance_eligibility` Lambda function
- Add a Gateway and connect the Lambda to it, with two CLI commands
- Give your agent the Gateway's tools and test the 74% / 75% edge cases

## Settings

✏️ Change `STUDENT_ID` to your own ID before running anything else.

In [ ]:
# Settings for this lab
import sys, json, time, uuid, pathlib, subprocess, shutil
import boto3

# ✏️ Put YOUR student ID here (the same one in every notebook). Lower-case letters and digits, e.g. priya07
STUDENT_ID = "student01"

REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"         # Amazon Nova 2 Lite
CHEAP_MODEL_ID = "us.amazon.nova-2-lite-v1:0"
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"

PROJECT = f"campusmate{STUDENT_ID}"                    # your AgentCore project (created in notebook 05)
PROJECT_DIR = ROOT / PROJECT                           # the folder it lives in
AGENT_FILE = PROJECT_DIR / "app" / "agent" / "main.py" # your agent's code

def agentcore(*args, folder=None):
    """Run an AgentCore CLI command (the same one you would type in a terminal) and show its output."""
    process = subprocess.Popen([shutil.which("agentcore") or "agentcore", *args], cwd=folder or PROJECT_DIR,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        if "[SDK info]" not in line:
            print(line, end="")
    if process.wait() != 0:
        raise RuntimeError("The agentcore command failed. Read the messages above.")

print("Student:", STUDENT_ID, "| Project folder:", PROJECT_DIR)

## What you built in earlier labs

The AgentCore CLI knows everything in your project. We ask it, so there are no IDs to copy.

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
AGENT_ARN = resources["agent"]["arn"]
RUNTIME_ID = AGENT_ARN.split("/")[-1]
LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"      # your agent's logs and traces
for name, r in resources.items():
    print(f"{r['resourceType']:14s} {name:16s} {r.get('arn', '')}")

## MCP in five minutes

**MCP (Model Context Protocol)** is an open standard for connecting AI apps to tools. 🏫 Think of it as the **USB-C port for tools**:
any MCP *client* (an agent) can plug into any MCP *server* (a tool provider).

- `tools/list`: "what can you do?"
- `tools/call`: "do this, with these inputs"

**AgentCore Gateway is a managed MCP server in AWS.** Behind it you can put Lambda functions or APIs. Tool names get a prefix:
`<target>___<tool>`, for example `attendance___check_attendance_eligibility`.

**Who may call your Gateway?** We use `AWS_IAM`: every request must be signed with AWS credentials from your account (your agent's,
or yours). Nobody outside can call it.

## Step 1: the Lambda function

**AWS Lambda** runs one small function when called, and you pay only for the milliseconds it runs. Ours reads the fictional `students.json` and applies the 75% rule:

In [ ]:
print((ROOT / "projects/lambda/check_attendance_eligibility/handler.py").read_text()[:1600])

Create it (and the small role every Lambda function needs to write its logs):

In [ ]:
iam = boto3.client("iam")
lam = boto3.client("lambda", region_name=REGION)
LAMBDA_ROLE = f"campusmate-lambda-role-{STUDENT_ID}"

# Every Lambda function needs a role. This one only lets it write its own logs (an AWS-managed policy).
try:
    lambda_role_arn = iam.get_role(RoleName=LAMBDA_ROLE)["Role"]["Arn"]
except iam.exceptions.NoSuchEntityException:
    trust = {"Version": "2012-10-17", "Statement": [{"Effect": "Allow", "Principal": {"Service": "lambda.amazonaws.com"},
                                                     "Action": "sts:AssumeRole"}]}
    lambda_role_arn = iam.create_role(RoleName=LAMBDA_ROLE, AssumeRolePolicyDocument=json.dumps(trust))["Role"]["Arn"]
    iam.attach_role_policy(RoleName=LAMBDA_ROLE, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole")
    time.sleep(10)   # a new role takes a few seconds to be ready

def create_lambda(name, handler_file, extra_files=()):
    """Zip the handler (plus any data files) and create or update the Lambda function."""
    import io, zipfile
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w") as z:
        z.write(handler_file, "handler.py")
        for path in extra_files:
            z.write(path, pathlib.Path(path).name)
    try:
        lam.update_function_code(FunctionName=name, ZipFile=buf.getvalue())
        lam.get_waiter("function_updated_v2").wait(FunctionName=name)
    except lam.exceptions.ResourceNotFoundException:
        lam.create_function(FunctionName=name, Runtime="python3.12", Role=lambda_role_arn, Handler="handler.lambda_handler",
                            Code={"ZipFile": buf.getvalue()}, Architectures=["arm64"], Timeout=15)
        lam.get_waiter("function_active_v2").wait(FunctionName=name)
    return lam.get_function(FunctionName=name)["Configuration"]["FunctionArn"]

ATTENDANCE_LAMBDA = f"campusmate-attendance-{STUDENT_ID}"
attendance_arn = create_lambda(ATTENDANCE_LAMBDA, ROOT / "projects/lambda/check_attendance_eligibility/handler.py",
                               [DATA_DIR / "students.json"])
print("Lambda ready:", attendance_arn)
# Quick test, calling the Lambda directly:
out = lam.invoke(FunctionName=ATTENDANCE_LAMBDA, Payload=json.dumps({"student_id": "AIDD-DEMO-003", "course_code": "AIDD303"}))
print(json.loads(out["Payload"].read()))

## Step 2: add a Gateway, and the Lambda as its first target

(Terminal: `agentcore add gateway --name toolsGateway --authorizer-type AWS_IAM`, then `agentcore add gateway-target ...`.) The `tool_schema.json` file tells the Gateway the tool's name, description and inputs.

In [ ]:
if "toolsGateway" in (PROJECT_DIR / "agentcore" / "agentcore.json").read_text():
    print("toolsGateway is already in your project")
else:
    agentcore("add", "gateway", "--name", "toolsGateway", "--authorizer-type", "AWS_IAM")
    attendance_target = {
        "name": "attendance",
        "targetType": "lambdaFunctionArn",
        "lambdaFunctionArn": {"lambdaArn": attendance_arn,
                              "toolSchemaFile": str(ROOT / "projects/lambda/check_attendance_eligibility/tool_schema.json")},
    }
    agentcore("add", "gateway-target", "--gateway", "toolsGateway", "--target-configuration", json.dumps(attendance_target))

## Step 3: give your agent the Gateway's tools

Same file again, with the lines marked **`# NEW in Lab 7`**:
- `GATEWAY_URL`: after deploy, AgentCore gives your agent the Gateway's address automatically.
- `gateway_tools()`: an MCP client for the Gateway. Strands accepts it in `tools=[...]` like any other tool.

In [ ]:
%%writefile {AGENT_FILE}
import json
from pathlib import Path
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

DATA_DIR = Path(__file__).parent / "data"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"   # Amazon Nova 2 Lite


# ---- Your tools from notebook 04 (unchanged) ----
@tool
def get_course_info(query: str) -> str:
    """Look up AIDD courses by course code (e.g. AIDD303) or a keyword in the title/description (e.g. docker).

    Args:
        query: A course code or a keyword.
    """
    q = query.lower()
    courses = json.loads((DATA_DIR / "courses.json").read_text())["courses"]
    hits = [c for c in courses if q == c["code"].lower() or q in (c["title"] + " " + c["description"]).lower()]
    return json.dumps(hits[:5]) if hits else json.dumps({"found": False, "codes": [c["code"] for c in courses]})


@tool
def get_campus_policy(topic: str) -> str:
    """Return CampusMate rules for a topic: attendance, internal_assessments, semester_end_exams, fees, leave,
    re_evaluation, library, hackathons. Always use this instead of guessing a rule.

    Args:
        topic: one of the topics above
    """
    policies = json.loads((DATA_DIR / "campus_policies.json").read_text())["policies"]
    return json.dumps(policies.get(topic.lower().strip(), {"found": False, "topics": list(policies)}))


# ---- NEW in Lab 6: memory ----
import os
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

MEMORY_ID = os.getenv("MEMORY_STUDENTMEMORY_ID")   # AgentCore fills this in for you after `agentcore add memory`


def memory_for(session, student):
    """Save this conversation, and bring back what we already know about this student."""
    return AgentCoreMemorySessionManager(
        AgentCoreMemoryConfig(memory_id=MEMORY_ID, session_id=session, actor_id=student,
                              retrieval_config={"/users/{actorId}/facts": RetrievalConfig(top_k=5),
                                                "/summaries/{actorId}": RetrievalConfig(top_k=3)}),
        region_name="us-east-1")


# ---- NEW in Lab 7: tools from the Gateway ----
from strands.tools.mcp import MCPClient
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

GATEWAY_URL = os.getenv("AGENTCORE_GATEWAY_TOOLSGATEWAY_URL")   # AgentCore fills this in after `agentcore add gateway`


def gateway_tools():
    """Connect to your Gateway (an MCP server). Requests are signed with the agent's AWS identity."""
    return MCPClient(lambda: aws_iam_streamablehttp_client(endpoint=GATEWAY_URL + "/mcp",
                                                           aws_service="bedrock-agentcore", aws_region="us-east-1"))


# ---- AgentCore Runtime wrapper (Lab 5) ----
app = BedrockAgentCoreApp()
agents = {}   # one agent per conversation (session)


@app.entrypoint
def invoke(payload, context):
    student = payload.get("student_id", "AIDD-DEMO-001")
    session = context.session_id
    if session not in agents:
        agents[session] = Agent(
            model=BedrockModel(model_id=MODEL_ID, max_tokens=1200),
            tools=[get_course_info, get_campus_policy, gateway_tools()],   # NEW in Lab 7: + gateway tools
            session_manager=memory_for(session, student),   # NEW in Lab 6
            system_prompt=f"You are CampusMate for AIDD students, talking to student {student}. "
                          "Use tools for courses, rules and attendance; never invent rules. Be brief.",
        )
    answer = agents[session](payload["prompt"])
    return {"result": str(answer)}


if __name__ == "__main__":
    app.run()

## Step 4: deploy

Creates the Gateway, connects the Lambda, and updates your agent (2-3 minutes). The CLI also gives the Gateway permission to call your Lambda, and your agent permission to call the Gateway.

In [ ]:
agentcore("deploy")

In [ ]:
def deployed():
    """Ask the AgentCore CLI what your project has in AWS: {name: details}."""
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "status", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
    return {r["name"]: r for r in json.loads(out.stdout)["resources"]}

def ask(prompt, session_id, student="AIDD-DEMO-001"):
    """Send one message to your deployed agent (agentcore invoke) and return its answer."""
    payload = json.dumps({"prompt": prompt, "student_id": student})
    out = subprocess.run([shutil.which("agentcore") or "agentcore", "invoke", "--json", "--runtime", "agent",
                          "--session-id", session_id, "--payload", payload], cwd=PROJECT_DIR, capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stdout + out.stderr)
    return json.loads(json.loads(out.stdout)["body"])["result"]

resources = deployed()
AGENT_ARN = resources["agent"]["arn"]
RUNTIME_ID = AGENT_ARN.split("/")[-1]
LOG_GROUP = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"      # your agent's logs and traces
for name, r in resources.items():
    print(f"{r['resourceType']:14s} {name:16s} {r.get('arn', '')}")

## 🔬 Live check: talk MCP to the Gateway yourself

AIDD-DEMO-003 has **74%** in AIDD303 (not eligible); AIDD-DEMO-004 has **exactly 75%** in AIDD301 (eligible). Edge cases are where bugs hide.

In [ ]:
from mcp import ClientSession
from mcp_proxy_for_aws.client import aws_iam_streamablehttp_client

GATEWAY_URL = boto3.client("bedrock-agentcore-control", region_name=REGION).get_gateway(
    gatewayIdentifier=resources["toolsGateway"]["arn"].split("/")[-1])["gatewayUrl"] + "/mcp"

async def call_tool(tool=None, args=None):
    """Talk MCP to your Gateway: list its tools, or call one. Signed with YOUR AWS identity."""
    async with aws_iam_streamablehttp_client(endpoint=GATEWAY_URL, aws_service="bedrock-agentcore", aws_region=REGION) as (read, write, _):
        async with ClientSession(read, write) as s:
            await s.initialize()
            if tool is None:
                return [t.name for t in (await s.list_tools()).tools]
            try:
                r = await s.call_tool(tool, args)
                return ("DENIED: " if r.isError else "ALLOWED: ") + r.content[0].text[:200]
            except Exception as e:
                return f"DENIED: {e}"

print("tools/list ->", await call_tool())
for sid in ("AIDD-DEMO-003", "AIDD-DEMO-004"):
    print(f"\n{sid}:", await call_tool("attendance___check_attendance_eligibility", {"student_id": sid}))

## Step 5: ask your agent

In [ ]:
print(ask("Am I eligible for my AIDD303 exam?", f"lab07-{uuid.uuid4()}", "AIDD-DEMO-003"))

## ✍️ Your turn

AIDD-DEMO-005 has good attendance but **fees pending**. Ask the agent, as that student, whether they can get their hall ticket.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
STUDENT = "TODO"   # TODO: "AIDD-DEMO-005"
if STUDENT != "TODO":
    print(ask("Can I get my hall ticket?", f"mine-{uuid.uuid4()}", STUDENT))
else:
    print("Skipped: set STUDENT first.")

## ✅ Checkpoint

You are on track if:
- The direct Lambda test printed AIDD303 with `eligible_for_exam: False`
- `tools/list` shows `attendance___check_attendance_eligibility`
- The agent told AIDD-DEMO-003 they are 1% short in AIDD303

**Stuck or behind?** Copy from `the Step 3 cell (whole agent file); finished version: projects/campusmate_agent.py` and keep going. That's what it's there for.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `Unknown tool` | The deploy didn't finish, or the target name is different | Re-run Step 4 and read its messages |
| Lambda `InvalidParameterValueException ... role` | The new role wasn't ready yet | Wait 10 seconds and re-run Step 1 |
| `RuntimeError: asyncio.run() cannot be called` | Used asyncio.run inside Jupyter | In notebooks just `await` the call, as above |